# Compare exact-hit policies in vX

Ordinary annotation retains exact junction hits. Benchmarking can exclude them from the hit list with `--exclude-exact`. Both runs below use the same fixed synthetic reference and default vX score. The reference catalogue and its fitted evidence stay fixed.

In [ ]:
# Identify the installed CLI and print the fresh kernel environment.
import os, sys, subprocess, tempfile, json
from pathlib import Path
from importlib.metadata import version
os.environ["POLARS_MAX_THREADS"] = "1"
print("Python:", sys.version, "CLI interpreter:", sys.executable)
for package in ("vdjmatch", "vdjtools", "seqtree", "polars"):
    print(package, version(package))
# Open this notebook from the repository root or the examples directory.
examples = next((p for p in (Path.cwd()/"examples", Path.cwd())
                 if (p/"inputs/reference.tsv").is_file()), None)
if examples is None:
    raise RuntimeError("Use the downloaded complete examples bundle, not this notebook alone")
# Deterministic demonstration: no sampling or RNG seeds are used.
output = Path(tempfile.mkdtemp(prefix="vdjmatch-example-"))/"fresh"
output.mkdir()
print("Fresh output directory:", output)


In [ ]:
# Recalculate both hit-list policies through the installed CLI.
for name, flags in (("exact-retained", []), ("exact-excluded", ["--exclude-exact"])):
    subprocess.run(
        [sys.executable, "-m", "vdjmatch.cli", "annotate",
         str(examples / "inputs/cohort/sample_a.tsv"),
         "--vdjdb", str(examples / "inputs/reference.tsv"), "--threads", "4",
         "--output-prefix", str(output / name), *flags],
        check=True, timeout=120,
    )

In [ ]:
# Compare all dispositions on matched source identities.
import polars as pl
keys = ["sample_id", "source_sequence_id"]
fields = ["selected_epitope", "selected_mhc_a", "selected_mhc_b", "selected_mhc_class",
          "score", "selection_status"]
selections = {}
reference_hashes = set()
for name, excluded in (("exact-retained", False), ("exact-excluded", True)):
    manifest = json.loads((output / f"{name}.manifest.json").read_text())
    assert manifest["score_models"] == ["vx"]
    assert manifest["exact_hits_excluded"] is excluded
    reference_hashes.add(manifest["reference"]["input_sha256"])
    t = pl.read_csv(output / f"{name}.selection.tsv", separator="	")
    selections[name] = t.select(keys + fields)

assert len(reference_hashes) == 1
retained, excluded = selections.values()
comparison = retained.join(excluded, on=keys, how="full", coalesce=True,
                           validate="1:1", suffix="_excluded").sort(keys)
assert comparison.height == retained.height == excluded.height == 2
assert retained["selected_epitope"].to_list() == ["DEMO_A", "DEMO_A"]
assert comparison.select(pl.col("score").ne(pl.col("score_excluded")).all()).item()
print(comparison.select(*keys, "selected_epitope", "score", "selection_status",
                        "selected_epitope_excluded", "score_excluded",
                        "selection_status_excluded"))

Both source queries remain present under either hit policy. The same-target near matches remain eligible after both exact hits are removed from their respective hit lists. The score changes reflect that restricted search and carry no error-rate interpretation.

## Apply the matching calibration

If you need expected false calls per null query, use a vX bank produced with the same reference, species, lane, tie rule, exact-hit policy and null population. Keep the policy and calibration in the manifest when reporting results. The synthetic fixture above has no valid null bank.